# SoulX-FlashHead: Talking Head Generator

Go to **Settings** > **Accelerator** > **T4 GPU** before running. Ensure **Internet** is switched **ON** in the right sidebar.

In [ ]:
# @title 1. Setup Environment
import os
import sys
import shutil
import re

# 1. Cleanup & Clone fresh in /kaggle/working
kb_dir = '/kaggle/working/SoulX-FlashHead'
if os.path.exists(kb_dir):
    if os.path.exists(os.path.join(kb_dir, 'SoulX-FlashHead')):
        print("🧹 Cleaning up nested directories...")
        shutil.rmtree(kb_dir)

if not os.path.exists(kb_dir):
    print("📥 Cloning SoulX-FlashHead...")
    !git clone https://github.com/Soul-AILab/SoulX-FlashHead.git {kb_dir}

%cd {kb_dir}

!apt-get install -y ffmpeg -q

# 2. Fix Dependencies (Aggressive Pinning)
print("🔧 Installing dependencies...")
!pip install -q --upgrade "transformers==4.44.2" "sentencepiece==0.1.99" "protobuf<5.0.0" 
!pip install -q xformers
!pip install -q "gradio>=5.0.0" diffusers accelerate tqdm imageio easydict ftfy imageio-ffmpeg scikit-image loguru pyloudnorm decord librosa flask huggingface_hub
!pip install -q "xfuser>=0.4.3" yunchang distvae beautifulsoup4 einops

# 3. Transformers MT5 Patch
import transformers
try:
    from transformers import MT5Tokenizer
    print("✅ MT5Tokenizer correctly imported!")
except (ImportError, AttributeError):
    print("⚠️ MT5Tokenizer missing, applying monkey-patch...")
    from transformers import T5Tokenizer
    transformers.MT5Tokenizer = T5Tokenizer
    sys.modules['transformers'].MT5Tokenizer = T5Tokenizer

# 4. Patch facecrop.py (Corrected Multi-line Patch)
fc_path = "flash_head/utils/facecrop.py"
!git checkout {fc_path}
with open(fc_path, "r") as f: fc = f.read()

if "import json" not in fc: fc = fc.replace("import os", "import os\nimport json")

target_pattern = r'^(\s+)crop_face = face_image\.crop\(scaled_bbox\)'
match = re.search(target_pattern, fc, re.MULTILINE)
if match:
    print("🩹 Patching facecrop.py...")
    indent = match.group(1)
    # Using multi-line format to avoid IndentationError from one-liner with-blocks
    replacement = f"{indent}with open('/tmp/bbox.json', 'w') as bf:\n{indent}    json.dump({{'bbox': scaled_bbox, 'img_w': img_w, 'img_h': img_h}}, bf)\n{indent}crop_face = face_image.crop(scaled_bbox)"
    fc = fc.replace(match.group(0), replacement)
    with open(fc_path, "w") as f: f.write(fc)
else:
    print("⚠️ Could not find target line in facecrop.py!")

# 5. Face Handler (Fixed paths)
os.makedirs("flash_head/utils", exist_ok=True)
with open("flash_head/utils/cpu_face_handler.py", "w") as f:
    f.write('''import cv2\nimport numpy as np\nimport os\nfrom typing import Tuple, List\n\nclass CPUFaceHandler:\n    def __init__(self, model_selection: int = 1, min_detection_confidence: float = 0.0):\n        proto = "/kaggle/working/SoulX-FlashHead/deploy.prototxt"\n        caffe = "/kaggle/working/SoulX-FlashHead/res10_300x300_ssd_iter_140000.caffemodel"\n        self.use_dnn = False\n        import urllib.request\n        if not os.path.exists(proto):\n            try: urllib.request.urlretrieve("https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt", proto)\n            except: pass\n        if not os.path.exists(caffe):\n            try: urllib.request.urlretrieve("https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel", caffe)\n            except: pass\n        if os.path.exists(proto) and os.path.exists(caffe):\n            self.net = cv2.dnn.readNetFromCaffe(proto, caffe)\n            self.use_dnn = True\n        else:\n            cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"\n            self.cascade = cv2.CascadeClassifier(cascade_path)\n\n    def detect(self, image: np.ndarray) -> Tuple[List, List]:\n        bboxs, scores = [], []\n        img_h, img_w = image.shape[:2]\n        if self.use_dnn:\n            blob = cv2.dnn.blobFromImage(image, 1.0, (300, 300), (104.0, 177.0, 123.0))\n            self.net.setInput(blob)\n            detections = self.net.forward()\n            for i in range(detections.shape[2]):\n                confidence = float(detections[0, 0, i, 2])\n                if confidence > 0.5:\n                    bboxs.append([float(detections[0, 0, i, 3]), float(detections[0, 0, i, 4]), float(detections[0, 0, i, 5]), float(detections[0, 0, i, 6])])\n                    scores.append(confidence)\n        else:\n            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)\n            faces = self.cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))\n            for (x, y, w, h) in faces:\n                bboxs.append([x/img_w, y/img_h, (x+w)/img_w, (y+h)/img_h]); scores.append(1.0)\n        return bboxs, scores\n\n    def __call__(self, image: np.ndarray) -> Tuple[List, List]:\n        return self.detect(image)\n''')

print("✅ Setup complete")

In [ ]:
# @title 2. Download Models
from huggingface_hub import snapshot_download
import os

os.makedirs("/kaggle/working/models", exist_ok=True)

print("📥 Downloading SoulX-FlashHead (1.3B)... may take a few mins")
snapshot_download(
    repo_id="Soul-AILab/SoulX-FlashHead-1_3B",
    local_dir="/kaggle/working/models/SoulX-FlashHead-1_3B",
    local_dir_use_symlinks=False
)

print("📥 Downloading wav2vec2-base-960h...")
snapshot_download(
    repo_id="facebook/wav2vec2-base-960h",
    local_dir="/kaggle/working/models/wav2vec2-base-960h",
    local_dir_use_symlinks=False
)

print("✅ Models downloaded successfully!")

In [ ]:
# @title 3. Start Launch App with 9:16 Support
import os
import re

with open('gradio_app.py', 'r') as f:
    code = f.read()

code = code.replace('value="models/SoulX-FlashHead-1_3B"', 'value="/kaggle/working/models/SoulX-FlashHead-1_3B"')
code = code.replace('value="models/wav2vec2-base-960h"', 'value="/kaggle/working/models/wav2vec2-base-960h"')
code = code.replace('app.launch()', 'app.launch(share=True, debug=True)')

# 9:16 Compositing Logic
composite_func = """
def _composite_to_original(video_path, image_path):
    import cv2, json, numpy as np, subprocess
    if not os.path.exists('/tmp/bbox.json'): return video_path
    with open('/tmp/bbox.json', 'r') as f: data = json.load(f)
    x1, y1, x2, y2 = data['bbox']
    w, h = data['img_w'], data['img_h']
    if w/h > 0.8: return video_path 
    orig = cv2.imread(image_path)
    cap = cv2.VideoCapture(video_path)
    out_path = video_path.replace('.mp4', '_916.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, 25.0, (w, h))
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_resized = cv2.resize(frame, (x2-x1, y2-y1))
        canvas = orig.copy()
        canvas[y1:y2, x1:x2] = frame_resized
        out.write(canvas)
    cap.release(); out.release()
    final = video_path.replace('.mp4', '_final_916.mp4')
    subprocess.run(['ffmpeg', '-y', '-i', out_path, '-i', video_path, '-c', 'copy', '-map', '0:v:0', '-map', '1:a:0', final], capture_output=True)
    return final if os.path.exists(final) else out_path
"""

# Monkey-patch inject for generated app
patch = """
import sys, transformers
from transformers import T5Tokenizer
transformers.MT5Tokenizer = T5Tokenizer
sys.modules['transformers'].MT5Tokenizer = T5Tokenizer
"""

# Inject the function and call it
code = patch + "import json, cv2, subprocess, numpy as np\n" + composite_func + code
code = code.replace("return final_video_path", "return _composite_to_original(final_video_path, cond_image)")

with open('gradio_app_916.py', 'w') as f:
    f.write(code)

print("🚀 Launching with 9:16 Support...")
!python gradio_app_916.py